In [ ]:
# Instala la dependencia necesaria en el entorno del notebook
%pip install -q google-genai

import os
import time
from dotenv import load_dotenv 
from google import genai
from google.genai import types

load_dotenv()

# --- Configuración del cliente ---
# Se recomienda usar variables de entorno: os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

# Nombre del archivo local
FILE_PATH = "ud_04_teoria_completa.pdf"
STORE_NAME = "mi_biblioteca_avanzada"

# 1. Verificar si el archivo existe localmente antes de empezar
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"No se encontró el archivo: {FILE_PATH}")

# 2. Buscar si ya existe un store con ese nombre para no duplicar
existing_stores = client.file_search_stores.list()
file_search_store = next(
    (s for s in existing_stores if s.display_name == STORE_NAME), 
    None
)

if not file_search_store:
    print("Creando nuevo almacén...")
    file_search_store = client.file_search_stores.create(
        config={'display_name': STORE_NAME}
    )
else:
    print(f"Usando almacén existente: {file_search_store.name}")

# 3. Subida e indexación con configuración de fragmentación (Chunking)
print("Subiendo e indexando archivo...")
operation = client.file_search_stores.upload_to_file_search_store(
    file=FILE_PATH,
    file_search_store_name=file_search_store.name,
    config={
        'display_name': 'Manual de Usuario V1',
        'chunking_config': {
            'white_space_config': {
                'max_tokens_per_chunk': 500,
                'max_overlap_tokens': 50
            }
        }
    }
)

# Espera síncrona hasta que la operación de indexación termine
while not operation.done:
    print("Procesando indexación...")
    time.sleep(5)
    operation = client.operations.get(operation)

print("¡Todo listo! Realizando consulta...")

# 4. Consulta utilizando la herramienta FileSearch con reintentos
import time
from google.api_core.exceptions import ResourceExhausted
from google.api_core.exceptions import ClientError

max_retries = 3
retry_delay = 60  # segundos

for attempt in range(max_retries):
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash-lite",  # Actualizado a versión estable disponible
            contents="¿Cómo se resetea el dispositivo según el manual?",
            config=types.GenerateContentConfig(
                tools=[
                    types.Tool(
                        file_search=types.FileSearch(
                            file_search_store_names=[file_search_store.name]
                        )
                    )
                ]
            )
        )
        break  # Si éxito, salir del loop
    except (ResourceExhausted, ClientError) as e:
        if attempt < max_retries - 1:
            print(f"Cuota excedida. Reintentando en {retry_delay} segundos... (Intento {attempt + 1}/{max_retries})")
            time.sleep(retry_delay)
        else:
            print("Se alcanzó el límite máximo de reintentos. Por favor, verifica tu cuota de API.")
            print(f"Error: {str(e)}")
            raise

print("\n" + "="*50)
print(f"RESPUESTA IA: {response.text}")
print("="*50 + "\n")

# 5. Mostrar fuentes (Citas y fragmentos recuperados)
metadata = response.candidates[0].grounding_metadata
if metadata and metadata.grounding_chunks:
    print("FUENTES CONSULTADAS:")
    for chunk in metadata.grounding_chunks:
        if chunk.retrieved_context:
            text_snippet = chunk.retrieved_context.text[:150].replace('\n', ' ')
            print(f"- {text_snippet}...")

# 6. OPCIONAL: Borrar para limpiar (Solo si el store es temporal)
# client.file_search_stores.delete(name=file_search_store.name, config={'force': True})

Note: you may need to restart the kernel to use updated packages.
Usando almacén existente: fileSearchStores/mibibliotecaavanzada-qhscii1jp14c
Subiendo e indexando archivo...
Procesando indexación...
¡Todo listo! Realizando consulta...


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}